<a href="https://colab.research.google.com/github/gauravd12345/language_models/blob/main/seq2seq/seq2seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [107]:
import re
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

dataset = "hf://datasets/PaulineSanchez/Translation_words_and_sentences_english_french/data/train-00000-of-00001-3d14582ea46e1b17.parquet"
df = pd.read_parquet(dataset)

[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


device: cuda


In [108]:
c1, c2 = df.columns
enc = np.array(df[c1])
dec = np.array(df[c2])

for i in np.random.choice(np.arange(len(df)), 10):
  print(f"{enc[i]:<50} | {dec[i]}")

We are fully aware of the importance of the situation. | Nous sommes pleinement conscients de l'importance de la situation.
The teacher can see you.                           | L'instituteur peut te voir.
He took advantage of the opportunity.              | Il sauta sur l'occasion.
As we went around the corner, the lake came into view. | Alors que nous atteignions le coin, le lac nous apparut.
Don't make me go.                                  | Ne m'oblige pas à y aller.
I'd rather stay than go.                           | Je préférerais rester plutôt que de m'en aller.
I don't care if you're busy. Please help me now.   | Ça m'est égal que tu sois occupé. Aide-moi maintenant, je te prie !
He drank very little of the water.                 | Il but très peu de l'eau.
Did you go to any famous gardens?                  | Êtes-vous allés dans un des jardins célèbres?
Tom went home at 6:30.                             | Tom rentra à la maison à 18 h 30.


In [109]:
""" setup for encoder dataset """
enc_words = [word_tokenize(seq) for seq in enc]
enc_vocab = set()
for seq in enc_words:
  for word in seq:
    enc_vocab.add(word)

enc_vocab.add("<EOS>")
enc_vocab = sorted(enc_vocab)
enc_vocab_len = len(enc_vocab)

etoi, itoe = {}, {}
for i in range(enc_vocab_len):
    etoi[enc_vocab[i]] = i
    itoe[i] = enc_vocab[i]

print(f"english vocab size: {enc_vocab_len}")
print(enc_vocab[:10])

""" setup for decoder dataset """
dec_words = [word_tokenize(seq) for seq in dec]
dec_vocab = set()
for seq in dec_words:
  for word in seq:
    dec_vocab.add(word)

dec_vocab.add("<EOS>")
dec_vocab = sorted(dec_vocab)
dec_vocab_len = len(dec_vocab)

dtoi, itod = {}, {}
for i in range(dec_vocab_len):
    dtoi[dec_vocab[i]] = i
    itod[i] = dec_vocab[i]

print(f"\nfrench vocab size: {enc_vocab_len}")
print(dec_vocab[:10])

english vocab size: 16895
['!', '$', '%', '&', "'", "''", "'anticonstitutionnellement", "'bribery", "'d", "'em"]

french vocab size: 16895
['!', '$', '%', '&', "'", "''", "'Spot", "'achète", "'maison", "'oublie"]


In [110]:
""" Hyperparameters """

embedding_dim = 300
hidden_size = 300 # sequence length of the encoder output/decoder input vector
max_decoder_seq = 50

lr = 0.001
epochs = 10

enc_eos_tok = etoi["<EOS>"]
dec_eos_tok = dtoi["<EOS>"]

In [111]:
E = []
for sentence in enc:
  seq = []
  tok = word_tokenize(sentence) + ["<EOS>"]
  for word in tok:
    e_i = etoi[word]
    seq.append(e_i)
  E.append(torch.tensor(seq).to(device))

D = []
for sentence in dec:
  seq = []
  tok = word_tokenize(sentence) + ["<EOS>"]
  for word in tok:
    d_i = dtoi[word]
    seq.append(d_i)
  D.append(torch.tensor(seq).to(device))

for i in range(10): # tokenized english to french translations
  print(E[i], D[i])

tensor([1440,   19,  285], device='cuda:0') tensor([3832,    0,  232], device='cuda:0')
tensor([2414,    0,  285], device='cuda:0') tensor([942,   0, 232], device='cuda:0')
tensor([2414,    0,  285], device='cuda:0') tensor([939,   0, 232], device='cuda:0')
tensor([3046,  286,  285], device='cuda:0') tensor([3436,  233,  232], device='cuda:0')
tensor([3096,    0,  285], device='cuda:0') tensor([30686,  5230,     0,   232], device='cuda:0')
tensor([1187,    0,  285], device='cuda:0') tensor([  528, 14028,     0,   232], device='cuda:0')
tensor([1428,    0,  285], device='cuda:0') tensor([30684, 17160,     0,   232], device='cuda:0')
tensor([1611,   19,  285], device='cuda:0') tensor([3843,   15,  232], device='cuda:0')
tensor([2667,    0,  285], device='cuda:0') tensor([30686, 27911,     0,   232], device='cuda:0')
tensor([2667,    0,  285], device='cuda:0') tensor([4012,    0,  232], device='cuda:0')


In [112]:
class Seq2Seq(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc_embed = nn.Embedding(enc_vocab_len, embedding_dim)
        self.dec_embed = nn.Embedding(dec_vocab_len, embedding_dim)
        """ encoder RNN """
        self.E_x = nn.Linear(embedding_dim, hidden_size)
        self.E_h = nn.Linear(hidden_size, hidden_size)
        self.E_y = nn.Linear(hidden_size, hidden_size)

        """ decoder RNN """
        self.D_x = nn.Linear(embedding_dim, hidden_size)
        self.D_h = nn.Linear(hidden_size, hidden_size)
        self.D_y = nn.Linear(hidden_size, dec_vocab_len)

    def forward(self, w, h):
        # pseudocode assuming w is a single word instead of a tensor
        
        w = self.enc_embed(w)
        for word in w[:-1]:
          h = torch.tanh(self.E_x(word) + self.E_h(h))
          y = self.E_y(h)

        out = []

        w = self.dec_embed(torch.tensor(dec_eos_tok).to(device))
        h = y
        for _ in range(max_decoder_seq):
          h = torch.tanh(self.D_x(w) + self.D_h(h))

          y = self.D_y(h)
          out.append(y)
          prob = torch.softmax(y, dim=0)
          pred = prob.argmax().item()
        
          if pred == dec_eos_tok:
            break
          
          w = self.dec_embed(torch.tensor(pred).to(device))

        return torch.stack(out), h

seq2seq = Seq2Seq().to(device)

In [113]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(seq2seq.parameters(), lr=lr)

for epoch in range(epochs):
  total_loss = 0.0
  h = torch.ones(hidden_size).to(device)
  for e, d in zip(E[:200], D[:200]):
    h = h.detach()
    optimizer.zero_grad()
    
    out, h = seq2seq(e, h)
    min_len = min(len(out), len(d))
      
    out = out[:min_len]
    d = d[:min_len]
    loss = criterion(out, d)

    total_loss += loss.item()

    loss.backward()
    optimizer.step()
  
  print(f"Epoch: {epoch+1}/{epochs} | loss: {total_loss}")

Epoch: 1/10 | loss: 1352.717905163765
Epoch: 2/10 | loss: 842.0436059236526
Epoch: 3/10 | loss: 787.2393028736115
Epoch: 4/10 | loss: 722.3273748159409
Epoch: 5/10 | loss: 712.40478348732
Epoch: 6/10 | loss: 700.4232093095779
Epoch: 7/10 | loss: 710.4048953056335
Epoch: 8/10 | loss: 689.7136023044586
Epoch: 9/10 | loss: 678.8995115756989
Epoch: 10/10 | loss: 663.099847316742


In [114]:
test = "Tom noticed that his hands weren't clean."
seq = []
tok = word_tokenize(test) + ["<EOS>"]
for word in tok:
    e_i = etoi[word]
    seq.append(e_i)

h = torch.ones(hidden_size).to(device)
seq = torch.tensor(seq).to(device)

pred = seq2seq(seq, h)
for prob in pred:
    word = itod[torch.argmax(prob).item()]
    print(word, end=" ")

Je 165 